<a href="https://colab.research.google.com/github/BotLed/Road-Sign-Detection-Model/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
import os

os.makedirs('/content/datasets', exist_ok=True)
with zipfile.ZipFile('/content/archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/datasets/archive')

In [ ]:
from ultralytics import YOLO

# Load the model
model = YOLO('yolov8n.pt')

# Train
results = model.train(
    data='/content/datasets/archive/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    mosaic=1.0,
    mixup=0.1,
    name='colab_baseline_signs'
)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/archive/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=colab_baseline_signs2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

In [ ]:
import cv2
import os
import numpy as np
from pathlib import Path

def apply_digital_tape(image, bbox, color=(50, 50, 50), thickness_ratio=0.2):
    """
    Applies a horizontal tape strip across center of the bounding box.
    """
    h, w, _ = image.shape
    x_center, y_center, b_w, b_h = bbox

    # converts normalized YOLO coordinates to pixel coordinates
    x1 = int((x_center - b_w / 2) * w)
    y1 = int((y_center - b_h / 2) * h)
    x2 = int((x_center + b_w / 2) * w)
    y2 = int((y_center + b_h / 2) * h)

    # define tape dimensions (horizontal strip across middle)
    tape_h = int((y2 - y1) * thickness_ratio)
    tape_y1 = y_center * h - (tape_h / 2)
    tape_y2 = y_center * h + (tape_h / 2)

    # draw tape
    cv2.rectangle(image, (x1, int(tape_y1)), (x2, int(tape_y2)), color, -1)
    return image

def apply_adversarial_patch(image, bbox):
  h, w, _ = image.shape
  x_center, y_center, b_w, b_h = bbox

  # convert to pixel coordinates
  x1 = int((x_center - b_w / 4) * w)
  y1 = int((y_center - b_h / 4) * h)
  x2 = int((x_center + b_w / 4) * w)
  y2 = int((y_center + b_h / 4) * h)

  # gen adversarial noise
  noise_patch = np.random.randint(0, 255, (y2-y1, x2-x1, 3), dtype=np.uint8)

  # overlay the noise onto sign
  image[y1:y2, x1:x2] = noise_patch
  return image

def apply_heavy_occlusion(image, bbox):
  h, w, _ = image.shape
  x_center, y_center, b_w, b_h = bbox

  # increases tape patch size to 80 percent of bounding box
  x1 = int((x_center - b_w * 0.4) * w)
  y1 = int((y_center - b_h * 0.4) * h)
  x2 = int((x_center + b_w * 0.4) * w)
  y2 = int((y_center + b_h * 0.4) * h)

  cv2.rectangle(image, (x1, y1), (x2, y2), (128, 128, 128), -1)
  return image

val_images_path = '/content/datasets/archive/valid/images'
val_labels_path = '/content/datasets/archive/valid/labels'
output_path = '/content/datasets/archive/attacked_val/images'
os.makedirs(output_path, exist_ok=True)

# processes images
for img_name in os.listdir(val_images_path):
    if not img_name.endswith(('.jpg', '.jpeg', '.png')): continue

    img = cv2.imread(os.path.join(val_images_path, img_name))
    label_file = os.path.join(val_labels_path, img_name.rsplit('.', 1)[0] + '.txt')

    if os.path.exists(label_file):
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.split()
                # YOLO format
                bbox = [float(x) for x in parts[1:]]
                img = apply_adversarial_patch(img, bbox)

    cv2.imwrite(os.path.join(output_path, img_name), img)

print(f"Vandalized images saved to: {output_path}")

Vandalized images saved to: /content/datasets/archive/attacked_val/images


In [ ]:
# run validations on attacked data
results = model.val(data='/content/datasets/archive/attack.yaml', plots=True)

print(f"NEW mAP50: {results.box.map50}")

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2337.0±750.7 MB/s, size: 117.7 KB)
val: Scanning /content/datasets/archive/attacked_val/labels.cache... 39 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 39/39 14.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7it/s 1.8s
                   all         39         74      0.649       0.14      0.148      0.102
                     1          1          1          0          0      0.199      0.139
                   108          1          1          1          0     0.0128    0.00765
                   121          1          2          1          0       0.13      0.042
                   122          1          1          0          0          0          0
                   128          1          1       0.35          1      0.497      0.398
                   139      

In [ ]:
import os

original_labels = '/content/datasets/archive/valid/labels'
attacked_labels_folder = '/content/datasets/archive/attacked_val/labels'

os.makedirs(attacked_labels_folder, exist_ok=True)

# symlink all labels from valid to attacked_val, did this bs because labels were not linked
for label_file in os.listdir(original_labels):
    src = os.path.join(original_labels, label_file)
    dst = os.path.join(attacked_labels_folder, label_file)
    if not os.path.exists(dst):
        os.symlink(src, dst)

print("Labels linked successfully!")

Labels linked successfully!
